# B2-024-gpu-scientific-ml-capstone — Practice p11 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** mixture-parameter-regression

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

X5 = torch.tensor([0.0, 0.25, 0.5, 0.75, 1.0], dtype=torch.float64)
W = torch.tensor([[1.0, 0.5, 1.2]], dtype=torch.float64)
C = torch.tensor([[0.7, 0.2, 0.45]], dtype=torch.float64)
S = torch.tensor([[0.05, 0.08, 0.06]], dtype=torch.float64)
EXPECTED_F = torch.tensor([[2.1968466812e-02, 4.1592788537e-01, 8.4875530921e-01, 6.0653513172e-01, 1.5229979745e-08]],
                          dtype=torch.float64)
RAW = torch.tensor([[1.0, 0.7, 0.05, 0.5, 0.2, 0.08, 1.2, 0.45, 0.06]], dtype=torch.float64)
CANON = torch.tensor([[0.5, 0.2, 0.08, 1.2, 0.45, 0.06, 1.0, 0.7, 0.05]], dtype=torch.float64)
ATOL = 1e-10
RTOL = 1e-10

## Solution

**Mixture function.** With `x` of shape `(P,)` and parameters of shape `(N, 3)`, the broadcast `x[None, :, None] - centers[:, None, :]` has shape `(N, P, 3)`; each bump is $w\exp(-(x-c)^2/(2s^2))$ and the sum over the last axis gives `(N, P)`. All arithmetic stays in the inputs' dtype.

**Canonical order.** View `params` as `(N, 3, 3)` (component × (w, c, s)), compute a **stable** `argsort` of the centers per row, and `gather` whole triples with that order, so each triple travels intact. `gather` creates a new tensor, so the input is untouched.

In [ ]:
def mixture_function(x, weights, centers, widths):
    diff = x[None, :, None] - centers[:, None, :]                          # (N, P, 3)
    bumps = weights[:, None, :] * torch.exp(-diff ** 2 / (2 * widths[:, None, :] ** 2))
    return bumps.sum(dim=-1)


def canonicalize(params, key=1):
    p = params.reshape(-1, 3, 3)
    order = torch.argsort(p[..., key], dim=1, stable=True)               # key=1: centers
    return torch.gather(p, 1, order[..., None].expand(-1, -1, 3)).reshape(-1, 9)


def canonicalize_by_weight(params):
    # Wrong variant: orders the triples by weight
    return canonicalize(params, key=0)


def split(params):
    p = params.reshape(-1, 3, 3)
    return p[..., 0], p[..., 1], p[..., 2]


# Probe 1
F1 = mixture_function(X5, W, C, S)
# Probe 2
raw_before = RAW.clone()
canon_raw = canonicalize(RAW)
canon_canon = canonicalize(CANON)
# Probe 3
theta = capstone_data.train_rows("mixture")[1]
perm = [2, 0, 1]
theta_perm = theta.reshape(-1, 3, 3)[:, perm].reshape(-1, 9)
x32 = capstone_data.MIXTURE_X
f_orig = mixture_function(x32, *split(theta))
f_perm = mixture_function(x32, *split(theta_perm))
canon_theta = canonicalize(theta)
canon_perm = canonicalize(theta_perm)
# Probe 4
values, theta_v = capstone_data.val_rows("mixture")
f_val = mixture_function(x32, *split(theta_v))
f_ref = capstone_data.mixture_values(x32, theta_v)
resid_std = (values - f_val).std().item()
# Probe 5
wrong = canonicalize_by_weight(RAW)
print("F1", F1.tolist())
print("canon(RAW)", canon_raw.tolist(), "| by weight", wrong.tolist())
print("theta perm max |df|", (f_orig - f_perm).abs().max().item(), "| residual std", resid_std)

### Answer check

In [ ]:
# Probe 1
assert F1.shape == (1, 5) and F1.dtype == torch.float64
assert torch.allclose(F1, EXPECTED_F, atol=ATOL, rtol=RTOL)
# Probe 2
assert torch.equal(canon_raw, CANON) and torch.equal(canon_canon, CANON)
assert torch.equal(RAW, raw_before)
# Probe 3
assert canon_theta.shape == (800, 9) and torch.equal(canon_theta, theta)
assert canon_perm.shape == (800, 9) and torch.equal(canon_perm, theta)
assert not torch.equal(theta_perm, theta)
assert torch.allclose(f_perm, f_orig, atol=1e-6, rtol=1e-6)
# Probe 4
assert f_val.shape == (200, 32) and torch.allclose(f_val, f_ref, atol=1e-6, rtol=1e-6)
assert 0.015 <= resid_std <= 0.025
# Probe 5
assert not torch.equal(wrong, CANON)